Groundwater | Transport Modeling Track

# Step 4: Model Implementation — Building the Spill→Capture GWT Model

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → **4-Build** → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In [03t](03t_modflow_transport.ipynb) you translated the perceptual model into MODFLOW 6 language — the advection-dispersion-reaction equation, the Péclet/Courant accuracy criteria, and the GWT packages (with **SRC** as the source for a known-mass spill). Now we **build and run** the model: a contaminant spills upgradient of a GWHE doublet, and we read the breakthrough at the extraction (monitoring) well.

## The question — and the trap

A contaminant **spills upgradient** of a GWHE doublet running flow-only (clean injection, extraction). The extraction well is the **monitoring / compliance well**, and the question is blunt: **does the plume reach the well above the regulatory threshold, and when — or bypass it?**

Here is the trap. The plume in the source→well corridor is **narrow** — only as wide as transverse dispersion lets it spread ($\alpha_T = 1$ m). The inherited flow grid has ~50 m cells; when a cell is far larger than the spreading length, the scheme adds **numerical dispersion** that smears the plume — most damagingly **sideways** — and mis-reads the concentration along the centre-line the well draws from. So we **refine the source→well corridor** to ~10 m, which resolves the *longitudinal* breakthrough; the *transverse* width stays under-resolved (we return to that in §9).

> **You will SEE this model built.** Rather than one opaque call, this notebook assembles the model on the real **Limmat valley aquifer**, step by step — its boundary conditions, initial conditions, source, and dispersion — and shows the **actual FloPy** that builds each package. You **read and modify**; you do not build from scratch. The tested builder `build_srcpulse_demo(...)` then runs the very packages you read.

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~35–45 min |
| **Prerequisites** | Transport Steps 1–3 |

By the end you will be able to:

1. **See how a transport model is assembled on a real aquifer** — its boundary conditions, initial conditions, source, and dispersion — by reading and modifying real FloPy, not building from scratch
2. Read the **breakthrough** at the monitoring well and state a **threshold verdict** (reach? exceed? when?)
3. Check the model with a **mass balance** and a **solubility** guardrail
4. **Verify the transport engine** against an exact 2D analytical solution (arrival, spread, sorption, decay) — an *engine / unit* check, **not** a compliance decision
5. Interpret **Péclet / Courant** and **grid orientation** correctly, and know which claims the grid supports (→ particle tracking for the lateral question)

In [ ]:
# Setup
import sys, inspect, tempfile
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

import numpy as np
import matplotlib.pyplot as plt
import flopy
from flopy.plot import PlotMapView
from flopy.discretization import StructuredGrid

# The PUBLIC, teachable model-construction API (see _SUPPORT/src/transport_srcpulse_demo.py):
# load -> refine -> new_sim -> add_flow_model -> add_transport_model -> couple_and_run,
# composed (and run at production Courant sizing) by build_srcpulse_demo.
from transport_srcpulse_demo import (
    load_limmat_flow, refine_corridor, new_sim, add_flow_model, add_transport_model,
    build_srcpulse_demo, INJ_XY, ABS_XY, SPILL_UPGRADIENT_M, DOUBLET_Q,
)
from transport_verify_2d import build_and_run_2d_verification, plume_2d_instantaneous
from shared_functions import check_task_with_solution, create_multiple_choice

---
## 1 — This is the Limmat valley aquifer

Before any transport, meet the model. `load_limmat_flow()` returns the **05f-calibrated coarse Limmat valley** GWF model and its GIS — the model boundary, and the Limmat and Sihl rivers. We ride *this* calibrated flow field; the transport model is built on top of it. The spill is placed ~90 m **upgradient** of the extraction well, along the **local flow direction** read straight from the calibrated field.

In [ ]:
# Load the 05f-calibrated coarse Limmat valley flow model + its GIS (boundary, Limmat/Sihl).
cgwf, boundary, rivers, exe = load_limmat_flow()

# Place the spill ~90 m UPGRADIENT of the extraction well, along the LOCAL flow direction
# read from the calibrated flow field (the same rule the model builder uses internally).
mg0 = cgwf.modelgrid
xc0, yc0 = np.array(mg0.xcellcenters), np.array(mg0.ycellcenters)
spd0 = cgwf.output.budget().get_data(text="DATA-SPDIS")[0]
ia = int(np.argmin((xc0 - ABS_XY[0])**2 + (yc0 - ABS_XY[1])**2))
u_reg = np.array([spd0["qx"][ia], spd0["qy"][ia]]); u_reg = u_reg / np.hypot(*u_reg)
spill_xy = (ABS_XY[0] - SPILL_UPGRADIENT_M * u_reg[0], ABS_XY[1] - SPILL_UPGRADIENT_M * u_reg[1])

fig, ax = plt.subplots(figsize=(9, 7))
pmv = PlotMapView(model=cgwf, ax=ax)
pmv.plot_grid(lw=0.15, color="0.85")                       # the calibrated flow grid, real LV95 coords
boundary.boundary.plot(ax=ax, color="k", lw=1.3, zorder=3)
rivers.plot(ax=ax, color="steelblue", lw=2.0, zorder=3)
for nm in ["Limmat", "Sihl"]:
    sub = rivers[rivers["GEWAESSERNAME"] == nm]
    if len(sub):
        p = sub.geometry.iloc[len(sub) // 2].representative_point()
        ax.annotate(nm, (p.x, p.y), color="steelblue", fontsize=9, fontweight="bold", zorder=6)
ax.scatter(*INJ_XY, marker="v", s=110, color="seagreen", ec="k", zorder=5, label="doublet injection (+Q)")
ax.scatter(*ABS_XY, marker="^", s=140, color="firebrick", ec="k", zorder=5, label="extraction = compliance well (-Q)")
ax.scatter(*spill_xy, marker="*", s=260, color="darkorange", ec="k", zorder=6, label="spill (SRC), ~90 m upgradient")
ax.set_xlabel("Easting  LV95 [m]"); ax.set_ylabel("Northing  LV95 [m]")
ax.set_title("The Limmat valley aquifer — calibrated 05f flow model, doublet + spill")
ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print(f"Injection well  : ({INJ_XY[0]:.0f}, {INJ_XY[1]:.0f})")
print(f"Extraction well : ({ABS_XY[0]:.0f}, {ABS_XY[1]:.0f})   <- compliance / monitoring well")
print(f"Spill (SRC)     : ({spill_xy[0]:.1f}, {spill_xy[1]:.1f})   ~{SPILL_UPGRADIENT_M:.0f} m upgradient")

---
## 2 — The corridor grid

The inherited flow grid is ~50 m — far too coarse for a narrow plume. `refine_corridor(...)` refines the **spill→extraction corridor** to ~10 m (the refinement resides inside the function), and returns a **grid bundle**: the DISV mesh, the cell arrays, and the injection / extraction / source cell indices. Below we plot the refined **DISV mesh**, zoomed on the corridor, with the grid drawn in LV95 coordinates. `refine_corridor` also **re-solves the flow** on this refined grid — carrying the calibrated K and boundary conditions from §1 over unchanged (only the *grid* is refined), so fine-grid heads are available for what follows.

In [ ]:
# Refine the spill->extraction corridor to ~10 m (base grid is ~50 m).
grid = refine_corridor(cgwf, boundary, rivers)
mg = grid["modelgrid"]
sx, sy = grid["spill_xy"]
csz = grid["cellsize"]; corr = grid["corridor_mask"]

fig, ax = plt.subplots(figsize=(8, 7))
pmv = PlotMapView(modelgrid=mg, ax=ax)
pmv.plot_grid(lw=0.3, color="0.55")
ax.scatter(sx, sy, marker="*", s=260, color="darkorange", ec="k", zorder=6, label="spill (SRC)")
ax.scatter(*ABS_XY, marker="^", s=140, color="firebrick", ec="k", zorder=6, label="extraction / compliance well")
ax.scatter(*INJ_XY, marker="v", s=110, color="seagreen", ec="k", zorder=6, label="injection well")
xs = [sx, ABS_XY[0], INJ_XY[0]]; ys = [sy, ABS_XY[1], INJ_XY[1]]; pad = 130
ax.set_xlim(min(xs) - pad, max(xs) + pad); ax.set_ylim(min(ys) - pad, max(ys) + pad)
ax.set_xlabel("Easting  LV95 [m]"); ax.set_ylabel("Northing  LV95 [m]")
ax.set_title("Corridor-refined DISV mesh  (~10 m in the corridor, ~50 m base)")
ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

print(f"Total cells: {grid['ncpl']}    corridor cells: {int(corr.sum())}")
print(f"Cell size [m]  refined corridor: {csz[corr].min():.1f}-{csz[corr].max():.1f}    "
      f"base grid (median): ~{np.median(csz[~corr]):.0f}")

---
## 3 — Numerical accuracy: Péclet and Courant

`Pe ≤ 2` (cell size `≤ 2·α`) and `Cr ≤ 1` (time step `≤ Δx/v`) are **accuracy** criteria — with TVD the solution stays *stable* above them, but it **smears**. Refining the corridor to ~10 m brings the **longitudinal** `Pe_L` under 2, so the well breakthrough is trustworthy. The **transverse** `Pe_T` stays far above 2 at any feasible cell size (because `α_T = 1` m) — a limitation we look at carefully in §9.

In [ ]:
alpha_L, alpha_T = 10.0, 1.0    # LOCKED dispersivities [m]
csz_corr = grid["cellsize"][grid["corridor_mask"]]
PeL = csz_corr / alpha_L
PeT = csz_corr / alpha_T

print("Accuracy criteria (not stability -- TVD stays stable above them, but smears):")
print(f"  Longitudinal grid Peclet  Pe_L = {PeL.min():.1f}-{PeL.max():.1f}   (target <= 2  -> resolved on the refined corridor)")
print(f"  Transverse   grid Peclet  Pe_T = {PeT.min():.0f}-{PeT.max():.0f}    (target <= 2  -> NOT resolved; alpha_T = 1 m is too small for any feasible cell)")
print()
print("The corridor is refined to ~10 m so Pe_L <= 2: the LONGITUDINAL breakthrough at the")
print("well is trustworthy. The TRANSVERSE axis stays under-resolved at any affordable grid.")

### Checkpoint 1 — grid Péclet

In [ ]:
check_task_with_solution('task_t04_checkpoint_1')

---
## 4 — The flow model on the transport grid

The **calibration is already done**: the hydraulic-conductivity field and the boundary conditions come straight from the 05f-calibrated model you *loaded* in §1 — nothing here re-calibrates them. What happens here is **re-gridding and re-solving the flow**, not rebuilding a calibration. Transport needs the ~10 m corridor cells (§2), so the calibrated K and BCs are carried onto the refined grid and flow is solved there; MODFLOW 6 then solves flow and transport **together on that one grid**, so the flow model (GWF) is assembled as part of the same simulation as the transport model (GWT).

So the pipeline is: **load the calibrated model (§1) → re-grid + re-solve flow on the corridor mesh (§2) → build the coupled GWF here (§4) → add transport (§5).** (The §2 solve provides the refined-grid heads used as starting heads and checks; the coupled simulation built here still contains and solves its **own** GWF on that same grid, together with the GWT.)

Below we build the flow model with the **actual FloPy calls — read them, edit them, run them.** Each package is one real MF6 object:

- **DISV** — `flopy.mf6.ModflowGwfdisv`: the corridor-refined unstructured grid you just plotted.
- **NPF** — `flopy.mf6.ModflowGwfnpf`: hydraulic conductivity from the calibrated model.
- **IC / STO** — `flopy.mf6.ModflowGwfic` / `flopy.mf6.ModflowGwfsto`: initial heads; steady state.
- **RCHA** — `flopy.mf6.ModflowGwfrcha`: areal recharge.
- **CHD** — `flopy.mf6.ModflowGwfchd`: fixed heads at the valley edges (Limmat/Sihl-driven).
- **RIV** — `flopy.mf6.ModflowGwfriv`: Limmat and Sihl river leakage.
- **WEL ×2** — `flopy.mf6.ModflowGwfwel`: the geothermal doublet, **flow only** ($+Q$ inject, $-Q$ extract) — no solute rides these wells.

In [ ]:
# Build the flow model on the refined grid with the real FloPy calls — read and edit these.
# (This is exactly what the tested builder add_flow_model does; §6 runs the numbers via build_srcpulse_demo.)
_sim = new_sim(tempfile.mkdtemp(), pulse_days=30.0, total_days=120.0, nstp_per_period=1, exe=exe)
gp = grid['gridprops']
_gwf = flopy.mf6.ModflowGwf(_sim, modelname='gwf', save_flows=True, newtonoptions='NEWTON')
flopy.mf6.ModflowGwfdisv(_gwf, nlay=1, ncpl=grid['ncpl'], nvert=gp['nvert'], top=grid['top'],
                         botm=grid['botm'], vertices=gp['vertices'], cell2d=gp['cell2d'])
flopy.mf6.ModflowGwfnpf(_gwf, icelltype=1, k=grid['k'], save_flows=True, save_specific_discharge=True)
flopy.mf6.ModflowGwfic(_gwf, strt=np.maximum(grid['heads'], grid['botm'][0] + 0.01))
flopy.mf6.ModflowGwfsto(_gwf, steady_state={0: True, 1: True})
flopy.mf6.ModflowGwfrcha(_gwf, recharge=grid['rch'])
flopy.mf6.ModflowGwfchd(_gwf, stress_period_data=[(tuple(r['cellid']), float(r['head'])) for r in grid['chd']])
flopy.mf6.ModflowGwfriv(_gwf, stress_period_data=[(tuple(r['cellid']), float(r['stage']),
                        float(r['cond']), float(r['rbot'])) for r in grid['riv']])
flopy.mf6.ModflowGwfwel(_gwf, pname='injw', stress_period_data={0: [[(0, grid['inj_cell']), abs(DOUBLET_Q)]]})
flopy.mf6.ModflowGwfwel(_gwf, pname='absw', stress_period_data={0: [[(0, grid['ext_cell']), -abs(DOUBLET_Q)]]})
flopy.mf6.ModflowGwfoc(_gwf, head_filerecord='gwf.hds', budget_filerecord='gwf.cbc',
                       saverecord=[('HEAD', 'LAST'), ('BUDGET', 'LAST')])
print('GWF packages:', _gwf.get_package_list())

In [ ]:
# Inspect the ACTUAL inputs of the model you just built.
print('CHD  (fixed heads at valley edges):', len(_gwf.chd.stress_period_data.get_data(0)),
      'cells; e.g.', _gwf.chd.stress_period_data.get_data(0)[0])
print('RIV  (Limmat/Sihl leakage)        :', len(_gwf.riv.stress_period_data.get_data(0)), 'river cells')
print('WEL  doublet (flow only)          : inject', _gwf.get_package('injw').stress_period_data.get_data(0)[0],
      '| extract', _gwf.get_package('absw').stress_period_data.get_data(0)[0])

# Drift-guard: the inline model above must match the TESTED builder add_flow_model (which
# build_srcpulse_demo uses for the §6 headline numbers). If the illustrative code ever diverges
# from what actually runs, this fails on execution — it cannot drift silently.
_ref = add_flow_model(new_sim(tempfile.mkdtemp(), pulse_days=30.0, total_days=120.0,
                              nstp_per_period=1, exe=exe), grid)
assert _gwf.get_package_list() == _ref.get_package_list()
assert np.allclose(_gwf.npf.k.array, _ref.npf.k.array)
assert np.allclose(_gwf.ic.strt.array, _ref.ic.strt.array)
assert _gwf.chd.stress_period_data.get_data(0).tolist() == _ref.chd.stress_period_data.get_data(0).tolist()
assert _gwf.riv.stress_period_data.get_data(0).tolist() == _ref.riv.stress_period_data.get_data(0).tolist()
for _wp in ('injw', 'absw'):
    assert (_gwf.get_package(_wp).stress_period_data.get_data(0).tolist()
            == _ref.get_package(_wp).stress_period_data.get_data(0).tolist())
print('\u2713 inline flow model matches the tested add_flow_model builder (K, heads, CHD/RIV/WEL)')

> 🧭 **Where you use this.** In your own case study you don't type these FloPy calls — a **builder** assembles the same kind of coupled flow + transport model from your configuration file, and you set your scenario there. §4 and §5 show you what is *inside* that builder, so you can read it, adapt it, and debug it when a run misbehaves.

---
## 5 — The transport packages, made visible

The solute model is built by `add_transport_model(sim, gwf, grid, ...)`. Same idea — **render the real source, then build it live**:

- **IC** — `flopy.mf6.ModflowGwtic`: initial concentration $C = 0$ everywhere.
- **ADV** — `flopy.mf6.ModflowGwtadv`: advection, **TVD** scheme.
- **DSP** — `flopy.mf6.ModflowGwtdsp`: **dispersion**, $\alpha_L = 10$ m, $\alpha_T = 1$ m.
- **MST** — `flopy.mf6.ModflowGwtmst`: mobile storage / porosity ($n_e = 0.20$); **sorption and decay are OFF by default** ($R = 1$, $\lambda = 0$).
- **SRC** — `flopy.mf6.ModflowGwtsrc`: the spill, as a finite **mass** pulse **in grams per day**, ON in period 0 and OFF in period 1.
- **SSM** — `flopy.mf6.ModflowGwtssm`: couples the flow boundary flows to transport.

**The units chain — get this wrong and you are 1000× off.** The flow model runs in **metres / day**, so concentration in **mg/L $=$ g/m³**, which makes the SRC source term a **mass rate in grams per day**. A released mass $M$ [g] over a pulse $T$ [d] on $n$ source cells gives `smassrate = M / (n·T)` g/d. Put the mass in **kilograms** and every concentration comes out 1000× too high.

In [ ]:
# The REAL FloPy that builds the transport model (IC, MST, ADV, DSP, SSM, SRC).
print(inspect.getsource(add_transport_model))

In [ ]:
# Build the transport model LIVE on the SAME sim and inspect the spill source term.
_gwt = add_transport_model(_sim, _gwf, grid, mass_g=3.0e5, pulse_days=30.0)
print("GWT packages :", _gwt.get_package_list())
print("SRC spill, period 0 (mass loading g/d):", _gwt.src.stress_period_data.get_data(0))
print("SRC OFF in period 1 (pulse ended)     :", _gwt.src.stress_period_data.get_data(1))
print("DSP dispersivities: alpha_L =", float(np.max(_gwt.dsp.alh.array)),
      "m,  alpha_T =", float(np.max(_gwt.dsp.ath1.array)), "m")

---
### Checkpoint — predict before you run

You have now seen the whole model: the flow-only doublet, the corridor-refined grid, and the SRC spill ~90 m upgradient of the compliance well. **Before you run it in the next section, commit to a prediction.**

In [ ]:
create_multiple_choice('task_t04_checkpoint_predict')

---
## 6 — Run it: breakthrough at the monitoring well

Now the payoff. `build_srcpulse_demo(...)` composes exactly the builders you just read — load → refine → flow → transport → couple → run (internally a **pilot** solve reads the velocity and sizes the Courant time step, then a **production** solve) — and returns the diagnostics. The question is blunt: **does the plume reach the compliance well above the threshold, and when?**

In [ ]:
# Stated pedagogical regulatory threshold (mg/L = g/m3)
THRESHOLD_mgL = 3.0

# Build + run the spill->capture SRC-pulse demo -- the TESTED path (cached after the first run).
res = build_srcpulse_demo(mass_g=3.0e5, pulse_days=30.0, total_days=120.0, solubility_mgL=1000.0)

print(f"Doublet (flow-only)  : injection cell {res.inj_cell}, extraction/monitoring cell {res.ext_cell}")
print(f"Spill (SRC pulse)    : {res.mass_g:.0f} g over {res.pulse_days:.0f} d  ->  {res.smassrate_gpd:.0f} g/d")
print(f"  spill location     : ({res.spill_xy[0]:.0f}, {res.spill_xy[1]:.0f})  (upgradient of the extraction well)")
print(f"Grid Peclet          : Pe_L {res.PeL_min:.1f}-{res.PeL_max:.1f} (<= 2 OK)   Pe_T {res.PeT_min:.0f}-{res.PeT_max:.0f} (>> 2)")

In [ ]:
t, c = res.times, res.breakthrough
peak, t_peak = res.peak_mgL, res.arrival_day
exceed = c > THRESHOLD_mgL
t_first = float(t[np.argmax(exceed)]) if exceed.any() else None

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, c, color='firebrick', lw=2.2, label='concentration at the monitoring well')
ax.axhline(THRESHOLD_mgL, color='0.4', ls='--', lw=1.4, label=f'threshold = {THRESHOLD_mgL:g} mg/L')
if t_first is not None:
    ax.axvline(t_first, color='steelblue', ls=':', lw=1.4)
    ax.text(t_first, peak * 0.5, f'  first exceedance\n  ~day {t_first:.0f}',
            color='steelblue', fontsize=9, va='center')
ax.set_xlabel('time since spill [d]'); ax.set_ylabel('concentration [mg/L]')
ax.set_title('Breakthrough at the extraction (monitoring) well')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

verdict = ("REACHES the well and EXCEEDS the threshold" if exceed.any()
           else "reaches the well but stays BELOW the threshold")
print(f"Peak {peak:.2f} mg/L at day {t_peak:.0f}.   Verdict: {verdict}.")
if t_first is not None:
    print(f"First exceedance of {THRESHOLD_mgL:g} mg/L at ~day {t_first:.0f}.")

---
## 7 — Does the model conserve mass?

Two sanity checks before trusting any number. **Mass balance:** the SRC mass we put in must equal what leaves at the well + boundaries + what is still stored, to within a tiny numerical imbalance. **Solubility:** because SRC sets a *mass rate*, the emergent source-cell concentration must stay **below the contaminant's solubility** — otherwise it implies separate-phase NAPL, which this dissolved-transport model cannot represent.

In [ ]:
print("Mass balance (from the GWT budget) [g]:")
for k, v in res.mass_balance.items():
    print(f"  {k:<30} {v:14.4g}")
print()
print(f"Solubility guardrail: emergent source concentration {res.emergent_C_mgL:.1f} mg/L "
      f"vs solubility {res.solubility_mgL:.0f} mg/L  ->  "
      f"{'PASS' if res.solubility_ok else 'FAIL'} (x{res.solubility_margin:.1f} margin)")

---
## 8 — Verify the transport engine against an exact solution

Mass balance says the model didn't *lose* solute — but it doesn't prove the scheme put it in the *right place at the right time*. For that we sanity-check the **transport scheme itself** on a problem with an **exact answer**. We can't do that on the doublet (convergent flow has no closed-form solution), so we run the **same MF6 GWT scheme** on a clean **uniform-flow** problem where an analytical solution *does* exist: the **2D instantaneous point-mass release** (the course-lecture solution) — a spill spreading and advecting in uniform flow, with retardation $R$ and decay $\lambda$:

$$c(x,y,t)=\frac{M/(\phi_e R m)}{2\pi\sqrt{2D_Lt/R}\,\sqrt{2D_Tt/R}}\exp\!\left[-\frac{(x-\tfrac{u}{R}t)^2}{4D_Lt/R}-\frac{y^2}{4D_Tt/R}\right]e^{-\lambda t}$$

If the numerical plume matches this curve — arrival, longitudinal and transverse spread — the **engine is sound**, and any error in the doublet result is then about the **grid and geometry**, not the solver.

In [ ]:
# Run the SAME GWT scheme on a resolved uniform-flow grid and compare to the exact plume.
ver = build_and_run_2d_verification()   # defaults: fine grid, conservative (R=1, lambda=0)

print(f"Verification toy: uniform flow u={ver.u:.2f} m/d,  grid dx={ver.delr:.0f} dy={ver.delc:.0f} m "
      f"(Pe_L={ver.Pe_L:.1f}, Pe_T={ver.Pe_T:.1f}),  compared at t={ver.compare_time:.0f} d")
print(f"  peak position        : {ver.peak_pos_err*100:5.2f}% error")
print(f"  peak concentration   : {ver.peak_conc_err*100:5.2f}% error")
print(f"  longitudinal sigma_x : {ver.sigma_x_err*100:5.2f}% error")
print(f"  transverse   sigma_y : {ver.sigma_y_err*100:5.2f}% error")
print(f"  mass                 : {ver.mass_err*100:5.3f}% error")

fig, (axc, axt) = plt.subplots(1, 2, figsize=(12, 4.2))
c_an = plume_2d_instantaneous(ver.centreline_x - ver.x0, 0.0, ver.compare_time,
                              ver.M, ver.phi_e, ver.m, ver.R, ver.DL, ver.DT, ver.u, ver.lam)
axc.plot(ver.centreline_x, ver.centreline_c, color='firebrick', lw=2, label='MODFLOW 6 (numerical)')
axc.plot(ver.centreline_x, c_an, 'k--', lw=1.5, label='analytical (exact)')
axc.set_xlabel('x [m]'); axc.set_ylabel('c [mg/L]')
axc.set_title(f'Centre-line (y=0),  t={ver.compare_time:.0f} d'); axc.legend(); axc.grid(alpha=0.3)

ct_an = plume_2d_instantaneous(ver.x_cm_num - ver.x0, ver.transverse_y - ver.y0, ver.compare_time,
                               ver.M, ver.phi_e, ver.m, ver.R, ver.DL, ver.DT, ver.u, ver.lam)
axt.plot(ver.transverse_y, ver.transverse_c, color='firebrick', lw=2, label='numerical')
axt.plot(ver.transverse_y, ct_an, 'k--', lw=1.5, label='analytical')
axt.set_xlabel('y [m]'); axt.set_ylabel('c [mg/L]')
axt.set_title('Transverse profile through the peak'); axt.legend(); axt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Plan view of the SAME numerical field, WITH its grid overlaid
# (universal rule: every 2D map of model output shows the grid, in real/local coords).
sg = StructuredGrid(delr=np.full(ver.ncol, ver.delr), delc=np.full(ver.nrow, ver.delc))
fig, ax = plt.subplots(figsize=(9, 3.8))
pmv = PlotMapView(modelgrid=sg, ax=ax)
cm = pmv.plot_array(ver.conc_field, cmap="viridis")
pmv.plot_grid(lw=0.1, color="w", alpha=0.30)
ax.set_xlabel("x [m]  (local frame)"); ax.set_ylabel("y [m]")
ax.set_title(f"Numerical plume at t = {ver.compare_time:.0f} d  (grid overlaid)")
plt.colorbar(cm, ax=ax, label="c [mg/L]", shrink=0.9)
ax.set_aspect("equal")
plt.tight_layout(); plt.show()

### Reactions too — retardation and decay

The same solution carries **retardation** (the peak travels at the slowed velocity $u/R$) and **first-order decay** (mass falls as $e^{-\lambda t}$). Turning both on verifies the **reactive** terms MF6 applies for sorbing / decaying contaminants — the reason this track models reactive transport at all.

> **Scope of this check.** This is an **engine / unit** verification — turn on $R$ and $\lambda$ and confirm the MST package **shifts and damps** the curve exactly as the analytical equation predicts. Whether sorption or decay change the *compliance verdict* at the Limmat well is a separate, **application** question, taken up in [08t Rung A](08t_model_application.ipynb).

In [ ]:
# Turn ON reaction: retardation R=2 (sorption) and decay with a 100-day half-life.
verR = build_and_run_2d_verification(R=2.0, lam=np.log(2) / 100.0)

v_num = (verR.x_cm_num - verR.x0) / verR.compare_time
print(f"Reactive verification (R={verR.R:.0f}, half-life {np.log(2)/verR.lam:.0f} d):")
print(f"  retarded peak velocity : numerical {v_num:.3f} m/d  vs  u/R = {verR.u/verR.R:.3f} m/d "
      f"(un-retarded u = {verR.u:.2f})")
print(f"  peak-position error    : {verR.peak_pos_err*100:.2f}%")
print(f"  transverse sigma error : {verR.sigma_y_err*100:.2f}%")
print(f"  mass vs M*exp(-lambda t): {verR.mass_err*100:.2f}% error")
print("  -> sorption (retardation) and first-order decay are reproduced too.")

> **Engine verified.** On a resolved grid the MF6 GWT scheme reproduces the exact analytical plume — arrival, longitudinal *and* transverse spread, sorption, and decay — to a percent or two. The solver is trustworthy. So whatever we can and cannot claim from the doublet is about **grid and geometry**, which is exactly §9.

---
## 9 — What the grid supports — and the lateral question

The verification earns us trust in the **longitudinal** result: arrival, peak, and threshold-exceedance timing are reproduced to a percent or two. The **lateral** width is a different story — but the reason is subtler than "$Pe_T$ is large."

In fact the verification toy (run it with a coarse transverse grid) shows that on a **grid-aligned** flow, TVD adds *almost no* transverse numerical dispersion even at $Pe_T = 10$: the *transported* transverse spread is **grid-independent** (apart from a small sub-cell representation term — only ~4% on 10 m cells). So a large $Pe_T$ *by itself* does not ruin the lateral width. The doublet is harder for two specific reasons:

1. **The flow is oblique to the grid.** Streamlines converge radially toward the extraction well, crossing cell faces at an angle — exactly the condition that produces **cross-wind numerical dispersion**, which smears the plume sideways (the optional demo below makes this visible).
2. **The physical plume is sub-cell.** With $\alpha_T = 1$ m the true transverse half-width is a few metres — narrower than the ~10 m corridor cells — so even a perfect scheme cannot *represent* it.

Together these mean the doublet's **lateral width / contaminated area** is not trustworthy — not because the solver is wrong (we just verified it), but because of grid **orientation** and **resolution**. So don't claim a contaminated *area* or an exact concentration contour.

> **The right tool for the lateral / wellfield question is particle tracking.** A **capture zone** (MODFLOW 6 PRT) is *advective* and free of cross-wind numerical dispersion — it answers "*which* area's water reaches the well," which the smeared concentration field cannot. That's [08t Rung D](08t_model_application.ipynb).

### Optional demo — watch cross-wind numerical dispersion appear (and vanish)

To *see* mechanism (1), run the same clean 2D verification but with the uniform flow set at **45° to the grid**. The physical dispersion is unchanged (the analytical cross-flow spread is fixed), so any extra width is pure **numerical** cross-wind dispersion. On a coarse grid it is large; **refine the grid and it disappears** — proving it is a grid artefact, not physics. (Set `RUN_OBLIQUE_DEMO = False` to skip the ~10 s of runs.)

In [ ]:
RUN_OBLIQUE_DEMO = True   # set False to skip (~10 s of MF6 runs)

if RUN_OBLIQUE_DEMO:
    # Square domain sized so the diagonal (45 deg) plume stays clear of every boundary.
    # XT3D ON (xt3d_off=False) makes the *physical* dispersion tensor exact at any angle,
    # so the residual cross-flow spread is purely the ADVECTIVE cross-wind numerical dispersion.
    OBL = dict(flow_angle_deg=45.0, xL=160.0, y_half=80.0, x0=45.0, y0=-45.0,
               total_time=100.0, xt3d_off=False)
    cell_sizes = [8.0, 4.0, 2.0]                                   # square cells, flow at 45 deg
    runs = [build_and_run_2d_verification(delr=cs, delc=cs, **OBL) for cs in cell_sizes]
    aligned = build_and_run_2d_verification(delc=10.0, compare_time=100.0)   # aligned flow, coarse transverse
    sig_an = runs[0].sigma_eta_an

    print(f"Cross-flow spread sigma_eta  (analytical = {sig_an:.1f} m):")
    print("  flow at 45 deg to the grid -- REAL cross-wind numerical dispersion:")
    for cs, r in zip(cell_sizes, runs):
        print(f"    cell {cs:>4.0f} m : sigma_eta {r.sigma_eta_num:6.2f} m  ({r.sigma_eta_err*100:5.1f}% high)"
              f"   raw transported moment {r.sigma_eta_centres_num:6.2f} m")
    print(f"  flow ALIGNED with the grid (coarse 10 m): sigma_y {aligned.sigma_y_num:.2f} m "
          f"({aligned.sigma_y_err*100:.1f}% high)   raw moment {aligned.meta['sigma_y_centres_num']:.2f} m (grid-independent)")

    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.plot(cell_sizes, [r.sigma_eta_err*100 for r in runs], 'o-', color='firebrick', lw=2,
            label='flow oblique (45°) — cross-wind numerical dispersion')
    ax.axhline(aligned.sigma_y_err*100, color='steelblue', ls='--', lw=1.5,
               label='flow aligned (coarse) — sub-cell only')
    ax.set_xlabel('cell size [m]'); ax.set_ylabel('cross-flow $\\sigma$ error [%]')
    ax.set_title('Transverse smearing vs grid resolution')
    ax.invert_xaxis(); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print("\nOblique to the grid, coarse cells smear the plume sideways by tens of percent;")
    print("refining the grid removes it. It is NUMERICAL (grid orientation + size), not physical.")

In [ ]:
if RUN_OBLIQUE_DEMO:
    # See the cross-wind smearing DIRECTLY: the coarsest oblique run's plume, with its grid.
    r0 = runs[0]
    sg0 = StructuredGrid(delr=np.full(r0.ncol, r0.delr), delc=np.full(r0.nrow, r0.delc))
    fig, ax = plt.subplots(figsize=(6.6, 5.6))
    pmv = PlotMapView(modelgrid=sg0, ax=ax)
    cm = pmv.plot_array(r0.conc_field, cmap="magma")
    pmv.plot_grid(lw=0.15, color="w", alpha=0.30)
    ax.set_xlabel("x [m]  (local frame)"); ax.set_ylabel("y [m]")
    ax.set_title(f"Flow at 45 deg to a {r0.delr:.0f} m grid -- plume smeared across-flow (grid overlaid)")
    plt.colorbar(cm, ax=ax, label="c [mg/L]", shrink=0.85)
    ax.set_aspect("equal")
    plt.tight_layout(); plt.show()

### Checkpoint 3 — defensible threshold claims

In [ ]:
create_multiple_choice('task_t04_checkpoint_3')

---
## Summary and handoff

You built and ran the spill→capture GWT model, verified the transport engine against an exact solution, and confronted the core trade-off of transport modelling.

**What you're taking forward**

| Result | Trust it? |
|---|---|
| Transport engine (the GWT scheme) | ✅ verified vs the exact 2D analytical plume (arrival, spread, $R$, $\lambda$) |
| Receptor breakthrough / peak / arrival-time | ✅ resolved (`Pe_L ≤ 2` on the refined corridor) |
| Threshold-exceedance verdict at the well | ✅ defensible |
| Mass balance | ✅ closes to a tiny imbalance |
| Source concentration vs solubility | ✅ checked |
| Lateral plume width / contaminated area | ❌ grid-orientation + sub-cell artefact → use **particle tracking** (Step 8) |

**Key idea:** a **known-mass SRC pulse** gives a grid-independent, mass-conservative source; the **engine is verified** against the analytical solution, so errors are about **grid and geometry**; refine the corridor for the longitudinal breakthrough; and answer the lateral / wellfield question with **particle tracking**, not the smeared concentration field.

**Continue to:** [Step 5 — Calibration](./05t_calibration.ipynb) (a short bridge) → [Step 8 — Applications](./08t_model_application.ipynb).